# Synthetic Wake Word Mini-Dataset Generator
## NGI0 Commons Fund deliverable — WP2

This notebook generates a **20-sample synthetic wake-word mini-dataset** using
[edge-tts](https://github.com/rany2/edge-tts) (free, CPU-only, no API key).
The output is a persistent LJSpeech-style directory ready to feed into the
[ww-trainer](https://github.com/OpenVoiceOS/ww-trainer) training pipeline.

**What you will learn:**
1. How to synthesise labelled wake-word audio samples from text using multiple TTS voices
2. How a synthetic dataset is structured (positive samples + metadata)
3. How to generate phonetically confusable adversarial negatives for evaluation

This is the CPU-feasible subset of the full pipeline in `tts2ww_full_pipeline.ipynb`
(which additionally handles voice conversion augmentation requiring a GPU).

> Developed by TigreGotico for OpenVoiceOS, funded by the
> [NGI0 Commons Fund](https://nlnet.nl/project/OpenVoiceOS) / NLnet grant **101135429**.

**CI execution:** ✅ executed headlessly with real audio outputs.


## 0 · Configuration

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import os

WAKE_WORD = "hey mycroft"        # phrase to synthesise
N_POSITIVE = 20                  # positive samples to generate
N_ADVERSARIAL = 10               # confusable negatives to generate (text only)
LANG = "en"

# Where the dataset is written. It persists after the notebook finishes.
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "./synth_ww_output")

# edge-tts voice pool — use several voices for speaker diversity
VOICES = [
    "en-US-JennyNeural",
    "en-US-GuyNeural",
    "en-GB-SoniaNeural",
    "en-GB-RyanNeural",
    "en-AU-NatashaNeural",
]

print(f"Wake word : {WAKE_WORD!r}")
print(f"Positives : {N_POSITIVE}")
print(f"Voices    : {VOICES}")
print(f"Output    : {os.path.abspath(OUTPUT_DIR)}")

## 1 · Install / verify edge-tts

edge-tts calls the Microsoft Edge read-aloud service, so this notebook needs
outbound internet access. It also needs the `ffmpeg` binary on the system to
convert the MP3 stream to 16 kHz mono WAV.

In [ ]:
import subprocess, sys

try:
    import edge_tts
    print(f"edge-tts already installed: {edge_tts.__version__}")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "edge-tts>=7.2.8"])
    import edge_tts
    print(f"edge-tts installed: {edge_tts.__version__}")

## 2 · Synthesise positive samples

We cycle through the voice pool to produce `N_POSITIVE` utterances of the
wake-word phrase.  Each file is 16 kHz mono WAV.


In [ ]:
import asyncio, subprocess, tempfile
from pathlib import Path
from itertools import cycle

DATASET_DIR = Path(OUTPUT_DIR).expanduser().resolve()
POS_DIR = DATASET_DIR / "positives"
POS_DIR.mkdir(parents=True, exist_ok=True)

# MP3 downloads are throwaway; only the converted WAVs are part of the dataset.
SCRATCH_DIR = Path(tempfile.mkdtemp(prefix="synth_ww_"))

print(f"Dataset directory: {DATASET_DIR}")

async def synth(text: str, voice: str, out_mp3: Path):
    comm = edge_tts.Communicate(text, voice)
    await comm.save(str(out_mp3))

def mp3_to_wav(mp3: Path, wav: Path):
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(mp3), "-ar", "16000", "-ac", "1", str(wav)],
        check=True, capture_output=True,
    )

voice_cycle = cycle(VOICES)
generated = []

for i in range(N_POSITIVE):
    voice = next(voice_cycle)
    mp3 = SCRATCH_DIR / f"pos_{i:04d}.mp3"
    wav = POS_DIR / f"pos_{i:04d}.wav"
    asyncio.run(synth(WAKE_WORD, voice, mp3))
    mp3_to_wav(mp3, wav)
    mp3.unlink()
    size_kb = wav.stat().st_size // 1024
    generated.append({"file": wav.name, "voice": voice, "label": WAKE_WORD})
    print(f"  [{i+1:2d}/{N_POSITIVE}]  {wav.name}  {voice}  ({size_kb} KB)")

print(f"\n{len(generated)} positive samples generated.")

## 3 · Write metadata CSV

Two pipe-separated files are written next to the audio. `metadata.csv` follows
the LJSpeech layout, `id|text|text`, which is what TTS and wake-word tooling
expects. `voices.csv` records which voice produced each clip, as `id|voice`.

In [ ]:
import csv

metadata_path = DATASET_DIR / "metadata.csv"
voices_path = DATASET_DIR / "voices.csv"

with open(metadata_path, "w", newline="") as f, open(voices_path, "w", newline="") as g:
    meta = csv.writer(f, delimiter="|")
    voices = csv.writer(g, delimiter="|")
    for row in generated:
        stem = Path(row["file"]).stem
        meta.writerow([stem, row["label"], row["label"]])
        voices.writerow([stem, row["voice"]])

print(f"Metadata written: {metadata_path}")
print(f"Voices written  : {voices_path}")
print(f"Rows: {len(generated)}")

# Quick preview
with open(metadata_path) as f:
    for line in f.readlines()[:5]:
        print(" ", line.rstrip())

## 4 · Generate adversarial negatives (text only, CPU)

Adversarial negatives are phonetically confusable words/phrases.
We use a simple grapheme-edit augmenter (single insertion/deletion/substitution)
which is the same method used in the full `tts2ww` pipeline.

> **Note:** Synthesising audio for negatives follows the same pattern as positives.
> Omitted here for brevity — the text file is the input to the ww-trainer
> negative-mining step.


In [ ]:
import random
import string

random.seed(42)

def grapheme_edits(word: str, n: int = 10) -> list:
    results = set()
    chars = list(word)
    # substitutions
    for i in range(len(chars)):
        for c in "bcdfghjklmnpqrstvwxyz":
            candidate = chars[:i] + [c] + chars[i+1:]
            results.add("".join(candidate))
    # insertions
    for i in range(len(chars) + 1):
        for c in "aeiou":
            candidate = chars[:i] + [c] + chars[i:]
            results.add("".join(candidate))
    # deletions
    for i in range(len(chars)):
        candidate = chars[:i] + chars[i+1:]
        if len(candidate) > 2:
            results.add("".join(candidate))
    # filter: remove original and empties
    results.discard(word)
    results = [r for r in results if r.strip()]
    return random.sample(list(results), min(n, len(results)))

# Generate for each word in the wake word
all_adversarials = set()
for token in WAKE_WORD.split():
    edits = grapheme_edits(token, n=N_ADVERSARIAL // len(WAKE_WORD.split()))
    all_adversarials.update(edits)

# Save to text file
adv_path = DATASET_DIR / f"{WAKE_WORD.replace(' ', '_')}_adversarials.txt"
adv_list = sorted(all_adversarials)[:N_ADVERSARIAL]
adv_path.write_text("\n".join(adv_list) + "\n")

print(f"Adversarial negatives ({len(adv_list)}):")
for w in adv_list:
    print(f"  {w}")
print(f"\nSaved to: {adv_path}")


## 5 · Dataset summary

In [ ]:
from pathlib import Path
import os

wavs = sorted(POS_DIR.glob("*.wav"))
total_size_kb = sum(w.stat().st_size for w in wavs) // 1024
avg_size_kb = total_size_kb // len(wavs) if wavs else 0

print("=== Synthetic wake-word mini-dataset ===")
print(f"Wake word      : {WAKE_WORD!r}")
print(f"Positive WAVs  : {len(wavs)}")
print(f"Total size     : {total_size_kb} KB")
print(f"Avg per sample : {avg_size_kb} KB")
print(f"Voices used    : {len(VOICES)}")
print(f"Adversarials   : {len(adv_list)} text entries")
print()
print("Directory layout:")
for p in sorted(DATASET_DIR.rglob("*"))[:20]:
    rel = p.relative_to(DATASET_DIR)
    size = f"({p.stat().st_size // 1024} KB)" if p.is_file() else ""
    print(f"  {rel}  {size}")


## 6 · Next steps

The dataset stays on disk in `OUTPUT_DIR` (`./synth_ww_output` by default). Point
the wake-word trainer at the `positives/` directory inside it:

```bash
python -m ww_trainer train \
    --wake-word "hey mycroft" \
    --dataset-dir ./synth_ww_output/positives \
    --tier micro
```

The trainer package is not published on PyPI yet, so install it from a released
wheel once one is available. `kaggle_quickstart_ww.ipynb` walks through the full
GPU-accelerated training flow, and `tts2ww_full_pipeline.ipynb` covers the
heavier augmentation stages (voice conversion, AudioSet negatives).

In [ ]:
import shutil

# Only the MP3 scratch area is disposable — the dataset itself is kept.
shutil.rmtree(SCRATCH_DIR, ignore_errors=True)

print(f"Dataset ready: {DATASET_DIR}")
print(f"  positives/          {len(list(POS_DIR.glob('*.wav')))} WAV files")
print(f"  metadata.csv        LJSpeech style, id|text|text")
print(f"  voices.csv          id|voice")
print(f"  {adv_path.name}")